# LeetCode 3697 — Compute Decimal Representation bằng Python

Link đề: https://leetcode.com/problems/compute-decimal-representation/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def decimalRepresentation(self, n: int) -> List[int]:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tách số nguyên dương `n` thành tổng **ít thành phần nhất** của các "thành phần cơ số 10" (một chữ số `1..9` nhân một lũy thừa của `10`), rồi trả về theo thứ tự **giảm dần**.
2. Nắm phép **tách chữ số** bằng `% 10` và `// 10`, và khái niệm **giá trị vị trí** (place value) của từng chữ số.
3. Biết vì sao đáp án chính là "mỗi chữ số khác `0` cho đúng một thành phần" và vì sao không thể làm ít hơn.
4. Viết 4 cách: duyệt lũy thừa từ lớn xuống nhỏ, tách chữ số từ phải rồi đảo, duyệt chuỗi, và tham lam trừ dần; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên bỏ số `0`, quên đảo thứ tự, quên nhân giá trị vị trí, và dùng sai chỉ số lũy thừa.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Một **thành phần cơ số 10** (base-10 component) là số có dạng *một chữ số từ 1 đến 9 nhân với một lũy thừa không âm của 10*, ví dụ `7`, `30`, `500`. Số `537` hay `102` **không** phải thành phần cơ số 10.

Hãy biểu diễn `n` thành **tổng** các thành phần cơ số 10 sao cho **số thành phần ít nhất**, và trả về danh sách các thành phần theo thứ tự **giảm dần**.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `537` | `[500, 30, 7]` | `537 = 500 + 30 + 7`, không thể dùng ít hơn 3 thành phần |
| 2 | `102` | `[100, 2]` | `102` không phải thành phần cơ số 10 nên cần 2 thành phần: `100 + 2` |
| 3 | `6` | `[6]` | `6` đã là một thành phần cơ số 10 |

Constraints: `1 <= n <= 10^9`.

In [1]:
def is_component(x):
    # x có dạng d * 10^k với d trong 1..9 không?
    s = str(x)
    return s[0] != '0' and set(s[1:]) <= {'0'}

for x in [7, 30, 500, 537, 102, 6, 1000, 90]:
    print(x, "->", is_component(x))

# kiểm tra các ví dụ của đề theo định nghĩa
for n, out in [(537, [500, 30, 7]), (102, [100, 2]), (6, [6])]:
    print(n, out, "tổng =", sum(out), "| đều là thành phần:", all(is_component(c) for c in out))
    assert sum(out) == n and out == sorted(out, reverse=True)

7 -> True
30 -> True
500 -> True
537 -> False
102 -> False
6 -> True
1000 -> True
90 -> True
537 [500, 30, 7] tổng = 537 | đều là thành phần: True
102 [100, 2] tổng = 102 | đều là thành phần: True
6 [6] tổng = 6 | đều là thành phần: True


## 3. Kiến thức nền

### 3.1. Chữ số và giá trị vị trí (place value)

Số `537` gồm các chữ số `5`, `3`, `7`. Chữ số `5` đứng ở hàng trăm nên đóng góp `5 * 100 = 500`; `3` ở hàng chục đóng góp `3 * 10 = 30`; `7` ở hàng đơn vị đóng góp `7 * 1 = 7`. Đó chính là "giá trị vị trí": `chữ số * 10^(vị trí tính từ phải, bắt đầu 0)`.

In [2]:
n = 537
print("537 = 5*100 + 3*10 + 7*1 =", 5 * 100 + 3 * 10 + 7 * 1)
for pos, d in enumerate(reversed(str(n))):
    print(f"vị trí {pos} (từ phải): chữ số {d}, giá trị vị trí 10^{pos} = {10 ** pos}, đóng góp {int(d) * 10 ** pos}")

537 = 5*100 + 3*10 + 7*1 = 537
vị trí 0 (từ phải): chữ số 7, giá trị vị trí 10^0 = 1, đóng góp 7
vị trí 1 (từ phải): chữ số 3, giá trị vị trí 10^1 = 10, đóng góp 30
vị trí 2 (từ phải): chữ số 5, giá trị vị trí 10^2 = 100, đóng góp 500


### 3.2. Toán tử `%` và `//` để tách chữ số

- `n % 10` cho **chữ số hàng đơn vị** (phần dư khi chia 10).
- `n // 10` **bỏ** chữ số hàng đơn vị (chia nguyên).

Lặp lại hai phép này sẽ bóc từng chữ số **từ phải sang trái**. Python có `divmod(n, 10)` trả cả hai cùng lúc.

In [3]:
n = 537
print(n % 10, n // 10)           # 7 53
print(divmod(537, 10))           # (53, 7)
x = 537
digits = []
while x:
    x, d = divmod(x, 10)
    digits.append(d)
print("các chữ số từ phải sang trái:", digits)

7 53
(53, 7)
các chữ số từ phải sang trái: [7, 3, 5]


### 3.3. Lũy thừa `**` và chuyển số thành chuỗi

`10 ** k` là `10` mũ `k` (`10 ** 0 = 1`). `str(n)` biến số thành chuỗi để duyệt từng ký tự, `int(ch)` đổi ký tự số về số. `len(str(n))` là **số chữ số** của `n`.

In [4]:
print(10 ** 0, 10 ** 2, 10 ** 9)
s = str(537)
print(s, len(s), [int(ch) for ch in s])
print(len(str(10 ** 9)), "<- 10^9 có 10 chữ số")

1 100 1000000000
537 3 [5, 3, 7]
10 <- 10^9 có 10 chữ số


### 3.4. `list.append`, `reverse`, `reversed` và `enumerate`

`lst.append(x)` thêm cuối danh sách. `lst.reverse()` đảo **tại chỗ**; `lst[::-1]` tạo bản đảo mới; `reversed(lst)` trả iterator. `enumerate(seq)` cho cặp `(chỉ số, phần tử)`.

In [5]:
lst = [7, 30, 500]
print(lst[::-1], list(reversed(lst)))
lst.reverse()
print(lst)
for i, ch in enumerate("537"):
    print(i, ch)

[500, 30, 7] [500, 30, 7]
[500, 30, 7]
0 5
1 3
2 7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Viết `537` theo giá trị vị trí: `537 = 500 + 30 + 7`. Mỗi chữ số khác `0` đã là **một thành phần** hoàn chỉnh khi nhân với lũy thừa `10` của nó. Còn chữ số `0` đóng góp `0` nên **bỏ qua**.

Với `102`: chữ số là `1`, `0`, `2` -> `100`, `0` (bỏ), `2` -> `[100, 2]`.

Câu hỏi "vì sao không thể ít hơn?" Mỗi thành phần chỉ có **một** chữ số khác `0`. Ý chính: cộng thêm một thành phần chỉ làm số chữ số khác `0` tăng tối đa 1, nên cần ít nhất bằng ấy thành phần; mục 10 sẽ chứng minh gọn.

### Bảng tính tay

| n | chữ số (trái -> phải) | thành phần khác 0 | kết quả |
|---|---|---|---|
| `537` | 5, 3, 7 | 500, 30, 7 | `[500, 30, 7]` |
| `102` | 1, 0, 2 | 100, 2 | `[100, 2]` |
| `6` | 6 | 6 | `[6]` |
| `4090` | 4, 0, 9, 0 | 4000, 90 | `[4000, 90]` |
| `1000000000` | 1 rồi chín số 0 | 1000000000 | `[1000000000]` |

Trực giác: số thành phần = **số chữ số khác 0** của `n`.

In [6]:
for n in [537, 102, 6, 4090, 10 ** 9]:
    s = str(n)
    comps = [int(ch) * 10 ** (len(s) - 1 - i) for i, ch in enumerate(s) if ch != '0']
    print(n, "->", comps, "| số chữ số khác 0 =", sum(ch != '0' for ch in s))

537 -> [500, 30, 7] | số chữ số khác 0 = 3
102 -> [100, 2] | số chữ số khác 0 = 2
6 -> [6] | số chữ số khác 0 = 1
4090 -> [4000, 90] | số chữ số khác 0 = 2
1000000000 -> [1000000000] | số chữ số khác 0 = 1


## 5. Cách 1 — Brute Force / baseline

Thử **mọi lũy thừa của 10** từ lớn nhất có thể (`10^9`) xuống `10^0`: với mỗi lũy thừa `p`, chữ số tại hàng đó là `(n // p) % 10`; nếu khác `0` thì thêm `digit * p` vào kết quả. Vì ta duyệt lũy thừa từ lớn xuống nhỏ nên kết quả đã **giảm dần**, không cần đảo.

Nút thắt: luôn lặp 10 lần dù `n` nhỏ (ví dụ `n = 6`), và phải biết trước cận trên `10^9`. Vẫn đúng và rất đơn giản.

In [7]:
class SolutionBrute:
    def decimalRepresentation(self, n):
        res = []
        for k in range(9, -1, -1):          # k = 9, 8, ..., 0  (n <= 10^9 nên tối đa 10 chữ số)
            p = 10 ** k
            digit = (n // p) % 10
            if digit != 0:
                res.append(digit * p)
        return res

print(SolutionBrute().decimalRepresentation(537), SolutionBrute().decimalRepresentation(102), SolutionBrute().decimalRepresentation(6))
print(SolutionBrute().decimalRepresentation(10 ** 9))

[500, 30, 7] [100, 2] [6]
[1000000000]


**Độ phức tạp:** Time `O(1)` (đúng 10 vòng, hay `O(D)` với `D` là số chữ số tối đa cho phép), Space `O(1)` ngoài kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đáp án chính là danh sách `chữ số * giá trị vị trí` của **mọi chữ số khác 0**, đọc từ trái sang phải.

**Quy tắc "nếu… thì…":**

- Nếu gặp chữ số `0` -> **bỏ qua** (nó đóng góp `0`, thêm vào chỉ làm tăng số thành phần và còn không hợp lệ vì `0` không phải thành phần).
- Nếu gặp chữ số `d != 0` ở vị trí `pos` (tính từ phải) -> thêm `d * 10^pos`.
- Muốn kết quả giảm dần: duyệt từ **chữ số cao nhất**; nếu bóc từ phải sang trái (bằng `%`, `//`) thì nhận kết quả **tăng dần** nên cuối cùng phải **đảo**.

**Invariant khi bóc từ phải:** sau mỗi bước, phần đã bóc cộng với `x * 10^pos` (phần còn lại) luôn bằng `n` ban đầu.

In [8]:
# Kiểm chứng ý tưởng: số thành phần = số chữ số khác 0, tổng = n
import random
random.seed(1)
for _ in range(5):
    n = random.randint(1, 10 ** 9)
    s = str(n)
    comps = [int(ch) * 10 ** (len(s) - 1 - i) for i, ch in enumerate(s) if ch != '0']
    assert sum(comps) == n and len(comps) == sum(ch != '0' for ch in s)
    print(n, comps)

144272510 [100000000, 40000000, 4000000, 200000, 70000, 2000, 500, 10]
611178003 [600000000, 10000000, 1000000, 100000, 70000, 8000, 3]
909925048 [900000000, 9000000, 900000, 20000, 5000, 40, 8]
861425549 [800000000, 60000000, 1000000, 400000, 20000, 5000, 500, 40, 9]
820096754 [800000000, 20000000, 90000, 6000, 700, 50, 4]


## 7. Các cách tiếp theo

### Cách 2 — Tách chữ số từ phải bằng `%`/`//`, rồi đảo

Giữ biến `place = 1` (giá trị vị trí). Mỗi vòng: `d = n % 10`; nếu `d != 0` thì `res.append(d * place)`; rồi `n //= 10` và `place *= 10`. Vì đi từ hàng đơn vị lên nên `res` đang **tăng dần** -> `res.reverse()` ở cuối. Cách này không cần biết trước cận `10^9` và dừng ngay khi `n` về `0`.

In [9]:
class SolutionDigits:
    def decimalRepresentation(self, n):
        res = []
        place = 1
        while n:
            d = n % 10
            if d:
                res.append(d * place)
            n //= 10
            place *= 10
        res.reverse()
        return res

print(SolutionDigits().decimalRepresentation(537), SolutionDigits().decimalRepresentation(102), SolutionDigits().decimalRepresentation(6))

[500, 30, 7] [100, 2] [6]


**Độ phức tạp:** Time `O(d)` với `d = log10(n)` là số chữ số, Space `O(d)` cho kết quả.

### Cách 3 — Duyệt chuỗi từ trái sang phải

Đổi `n` thành chuỗi `s`. Với ký tự thứ `i`, giá trị vị trí là `10 ** (len(s) - 1 - i)`. Duyệt từ trái sang phải nên kết quả tự giảm dần, chỉ cần lọc `'0'`. Dễ đọc nhất nhưng tốn bộ nhớ nhỏ cho chuỗi.

In [10]:
class SolutionString:
    def decimalRepresentation(self, n):
        s = str(n)
        size = len(s)
        return [int(ch) * 10 ** (size - 1 - i) for i, ch in enumerate(s) if ch != '0']

print(SolutionString().decimalRepresentation(537), SolutionString().decimalRepresentation(102), SolutionString().decimalRepresentation(6))

[500, 30, 7] [100, 2] [6]


**Độ phức tạp:** Time `O(d^2)` về mặt lý thuyết nếu tính `10 ** k` bằng số lớn, nhưng `d <= 10` nên coi là `O(d)`; Space `O(d)`.

### Cách 4 — Tham lam: lấy thành phần lớn nhất rồi trừ đi

Lấy `p = 10 ** (len(str(n)) - 1)` là lũy thừa cao nhất không vượt `n`. Thành phần lớn nhất không vượt `n` là `(n // p) * p`. Thêm vào kết quả rồi `n -= component`; lặp đến khi `n = 0`. Cách nghĩ này cũng chứng minh trực tiếp tính tối ưu: mỗi lần trừ ta **xóa đúng một chữ số khác 0** của `n`.

In [11]:
class SolutionGreedy:
    def decimalRepresentation(self, n):
        res = []
        while n:
            p = 10 ** (len(str(n)) - 1)
            comp = (n // p) * p
            res.append(comp)
            n -= comp
        return res

print(SolutionGreedy().decimalRepresentation(537), SolutionGreedy().decimalRepresentation(102), SolutionGreedy().decimalRepresentation(6))

[500, 30, 7] [100, 2] [6]


**Độ phức tạp:** Time `O(d^2)` (mỗi vòng gọi `len(str(n))`), thực tế `O(d)` vì `d <= 10`; Space `O(d)`.

## 8. TRACE từng bước

Trace `n = 4090` bằng **Cách 2** (bóc chữ số từ phải) rồi đảo:

| bước | n trước | d = n % 10 | place | thêm vào res | n sau |
|---|---|---|---|---|---|
| 1 | 4090 | 0 | 1 | (bỏ qua) | 409 |
| 2 | 409 | 9 | 10 | 90 | 40 |
| 3 | 40 | 0 | 100 | (bỏ qua) | 4 |
| 4 | 4 | 4 | 1000 | 4000 | 0 |

`res = [90, 4000]` -> đảo -> `[4000, 90]`.

In [12]:
def trace_digits(n):
    print("n =", n)
    res, place, step = [], 1, 0
    while n:
        step += 1
        d = n % 10
        print(f"bước {step}: n={n:>5} d={d} place={place:>5}", end=" ")
        if d:
            res.append(d * place)
            print("-> thêm", d * place, "| res =", res)
        else:
            print("-> bỏ qua (chữ số 0)")
        n //= 10
        place *= 10
    res.reverse()
    print("sau khi đảo:", res)
    return res

assert trace_digits(4090) == [4000, 90]
assert trace_digits(102) == [100, 2]

n = 4090
bước 1: n= 4090 d=0 place=    1 -> bỏ qua (chữ số 0)
bước 2: n=  409 d=9 place=   10 -> thêm 90 | res = [90]
bước 3: n=   40 d=0 place=  100 -> bỏ qua (chữ số 0)
bước 4: n=    4 d=4 place= 1000 -> thêm 4000 | res = [90, 4000]
sau khi đảo: [4000, 90]
n = 102
bước 1: n=  102 d=2 place=    1 -> thêm 2 | res = [2]
bước 2: n=   10 d=0 place=   10 -> bỏ qua (chữ số 0)
bước 3: n=    1 d=1 place=  100 -> thêm 100 | res = [2, 100]
sau khi đảo: [100, 2]


In [13]:
def trace_greedy(n):
    print("n =", n)
    res = []
    while n:
        p = 10 ** (len(str(n)) - 1)
        comp = (n // p) * p
        print(f"n={n:>5} p={p:>5} thành phần lớn nhất = {comp:>5}", end=" ")
        n -= comp
        res.append(comp)
        print("-> còn lại n =", n)
    return res

assert trace_greedy(537) == [500, 30, 7]

n = 537
n=  537 p=  100 thành phần lớn nhất =   500 -> còn lại n = 37
n=   37 p=   10 thành phần lớn nhất =    30 -> còn lại n = 7
n=    7 p=    1 thành phần lớn nhất =     7 -> còn lại n = 0


## 9. Code nộp LeetCode

In [14]:
from typing import List, Optional

class Solution:
    def decimalRepresentation(self, n: int) -> List[int]:
        res = []
        place = 1
        while n:
            d = n % 10
            if d:
                res.append(d * place)
            n //= 10
            place *= 10
        res.reverse()
        return res

## 10. Vì sao thuật toán đúng

- **Đúng (hợp lệ):** mỗi phần tử là `d * 10^pos` với `d` từ `1..9`, đúng định nghĩa thành phần cơ số 10. Tổng chúng bằng `sum(d_i * 10^i) = n` (khai triển thập phân), vì chữ số `0` đóng góp `0`.
- **Tối ưu (ít nhất):** gọi `z(x)` là số chữ số khác `0` của `x`. Cộng thêm **một** thành phần `d * 10^k` vào `x` làm `z` tăng **tối đa 1**: chữ số hàng `k` thành `x_k + d`; nếu `< 10` thì chỉ một chữ số đổi (tăng `z` tối đa 1); nếu `>= 10` thì hàng `k` về `x_k + d - 10` (mà `x_k >= 1` nên không tạo thêm chữ số khác 0), các chữ số `9` liên tiếp phía trên thành `0` (làm `z` giảm), và chữ số dừng cuối tăng thêm 1 (tăng `z` tối đa 1). Vì `z(0) = 0`, tổng của `m` thành phần có `z <= m`. Do đó `m >= z(n)`, mà đáp án đạt đúng `z(n)`.
- **Thứ tự:** các thành phần của đáp án thuộc các hàng khác nhau nên đều **phân biệt**, và sắp theo hàng giảm dần đúng là thứ tự giảm dần.
- **Duy nhất:** các thành phần của đáp án thuộc các hàng khác nhau và mỗi hàng đúng bằng chữ số của `n`, nên đáp án tối ưu theo cách này là xác định (đề cũng chỉ yêu cầu danh sách giảm dần này).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Duyệt 10 lũy thừa | `O(1)` (10 vòng) | `O(1)` ngoài kết quả |
| Tách chữ số từ phải + đảo | `O(d)` | `O(d)` |
| Duyệt chuỗi | `O(d)` | `O(d)` |
| Tham lam trừ dần | `O(d)` đến `O(d^2)` | `O(d)` |

`d = ⌊log10 n⌋ + 1` là số chữ số của `n` (`d <= 10`). Kết quả có tối đa `d` phần tử nên Space `O(d)` là cần thiết.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `n = 10^9`), và so sánh **tất cả cách** trên nhiều input; thêm kiểm tra tính chất: tổng bằng `n`, giảm dần, đúng dạng thành phần, số phần tử bằng số chữ số khác 0.

In [15]:
impls = [SolutionBrute(), SolutionDigits(), SolutionString(), SolutionGreedy(), Solution()]

cases = [
    (537, [500, 30, 7]), (102, [100, 2]), (6, [6]),
    (1, [1]), (10, [10]), (100, [100]), (4090, [4000, 90]),
    (999999999, [900000000, 90000000, 9000000, 900000, 90000, 9000, 900, 90, 9]),
    (10 ** 9, [10 ** 9]), (1010101, [1000000, 10000, 100, 1]),
]
for n, exp in cases:
    for impl in impls:
        got = impl.decimalRepresentation(n)
        assert got == exp, (type(impl).__name__, n, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 10 ca


In [16]:
import random
random.seed(3697)

def check_properties(n, comps):
    assert sum(comps) == n
    assert comps == sorted(comps, reverse=True) and len(set(comps)) == len(comps)
    assert all(is_component(c) for c in comps)
    assert len(comps) == sum(ch != '0' for ch in str(n))

for _ in range(500):
    n = random.randint(1, 10 ** 9)
    outs = [impl.decimalRepresentation(n) for impl in impls]
    assert all(o == outs[0] for o in outs), (n, outs)
    check_properties(n, outs[0])
print("500 test ngẫu nhiên (seed=3697): 5 cách giống nhau + 4 tính chất OK")

500 test ngẫu nhiên (seed=3697): 5 cách giống nhau + 4 tính chất OK


In [17]:
# Vét cạn nhỏ: n từ 1 đến 20000, và các số có nhiều chữ số 0
for n in range(1, 20001):
    r = Solution().decimalRepresentation(n)
    check_properties(n, r)
    assert r == SolutionString().decimalRepresentation(n) == SolutionGreedy().decimalRepresentation(n)
for n in [10 ** k for k in range(10)] + [2 * 10 ** k for k in range(9)]:
    assert Solution().decimalRepresentation(n) == [n]
print("vét cạn 1..20000 và lũy thừa của 10 OK")

vét cạn 1..20000 và lũy thừa của 10 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không bỏ chữ số 0

Thêm cả `0 * place = 0` vào kết quả sẽ sinh phần tử `0` (không phải thành phần cơ số 10, và làm tăng số phần tử).

In [18]:
def wrong_keep_zero(n):
    res, place = [], 1
    while n:
        res.append((n % 10) * place)        # SAI: không lọc chữ số 0
        n //= 10
        place *= 10
    res.reverse()
    return res

print(wrong_keep_zero(102), "!=", Solution().decimalRepresentation(102))
assert wrong_keep_zero(102) == [100, 0, 2] != Solution().decimalRepresentation(102)

[100, 0, 2] != [100, 2]


### Lỗi 2: quên đảo thứ tự

Bóc từ phải sang trái cho kết quả **tăng dần**, nhưng đề yêu cầu **giảm dần**.

In [19]:
def wrong_no_reverse(n):
    res, place = [], 1
    while n:
        d = n % 10
        if d:
            res.append(d * place)
        n //= 10
        place *= 10
    return res                               # SAI: thiếu res.reverse()

print(wrong_no_reverse(537), "!=", Solution().decimalRepresentation(537))
assert wrong_no_reverse(537) == [7, 30, 500] != Solution().decimalRepresentation(537)

[7, 30, 500] != [500, 30, 7]


### Lỗi 3: quên nhân giá trị vị trí

Chỉ lấy chữ số (`7, 3, 5`) mà không nhân `10^pos` thì tổng không còn bằng `n`.

In [20]:
def wrong_no_place(n):
    res = []
    while n:
        d = n % 10
        if d:
            res.append(d)                    # SAI: thiếu * place
        n //= 10
    return res[::-1]

print(wrong_no_place(537), "!=", Solution().decimalRepresentation(537))
assert wrong_no_place(537) == [5, 3, 7] != Solution().decimalRepresentation(537)

[5, 3, 7] != [500, 30, 7]


### Lỗi 4: dùng sai chỉ số lũy thừa khi duyệt chuỗi

Với `enumerate(s)` từ trái sang phải, lũy thừa phải là `len(s) - 1 - i`, **không phải** `i`.

In [21]:
def wrong_index_power(n):
    s = str(n)
    return [int(ch) * 10 ** i for i, ch in enumerate(s) if ch != '0']      # SAI: 10 ** i

print(wrong_index_power(537), "!=", Solution().decimalRepresentation(537))   # [5, 30, 700]
assert wrong_index_power(537) == [5, 30, 700] != Solution().decimalRepresentation(537)

[5, 30, 700] != [500, 30, 7]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói "tổng của các số dạng `chữ số × 10^k`" -> đó là **khai triển thập phân**; đáp án = chữ số khác 0 nhân giá trị vị trí.
- Gặp "tách chữ số" -> nghĩ ngay `% 10` và `// 10`, hoặc `str(n)`.
- Bóc từ phải thì kết quả ngược; nhớ `reverse()` hoặc `res[::-1]`.
- Muốn dễ nghĩ: viết ra `n = a*10^2 + b*10 + c` rồi bỏ số hạng bằng `0`.
- Với `n <= 10^9` có thể duyệt cứng 10 lũy thừa, rất khó sai.
- Tự kiểm tra nhanh: `sum(res) == n` và `res` giảm dần.

## 15. Biến thể của bài

1. **Số thành phần ít nhất cho mọi số trong đoạn** `[1, N]`: đếm tổng số chữ số khác 0.
2. **Cho phép thành phần có chữ số 0..9 và tối đa `k` thành phần.**
3. **Chuyển sang cơ số khác** (cơ số 2): biểu diễn `n` thành tổng các lũy thừa của 2 (chính là các bit 1).
4. **Roman numerals** (LeetCode 12): cũng tách theo hàng đơn vị, chục, trăm, nghìn.
5. **Add to Array-Form of Integer / Plus One**: thao tác trên từng chữ số của số lớn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Duyệt 10 lũy thừa | `(n // 10^k) % 10` cho `k = 9..0` | `O(1)` | `O(1)` | Cận trên đã biết, ít lỗi |
| Tách chữ số + đảo | `% 10`, `// 10`, rồi `reverse` | `O(d)` | `O(d)` | Cách chuẩn, nộp bài |
| Duyệt chuỗi | `enumerate(str(n))` | `O(d)` | `O(d)` | Dễ đọc, viết nhanh |
| Tham lam trừ dần | lấy thành phần lớn nhất rồi trừ | `O(d)` | `O(d)` | Giải thích tối ưu theo trực giác |

Nộp bài: cách 2 hoặc cách 3 đều gọn; cách 3 là một dòng.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().decimalRepresentation(7009)` ra gì?

**Bài 2 (tính tay):** biểu diễn `380250` thành các thành phần cơ số 10, giảm dần.

**Bài 3 (điền TODO):** hoàn thành `count_components(n)` trả về **số** thành phần ít nhất (không cần sinh danh sách).

**Bài 4 (tự viết lại):** viết `decimal_representation_str(n)` bằng đúng **một dòng** list comprehension với `enumerate` và `str`.

**Bài 5 (mini challenge):** viết `total_components(N)` trả về tổng số thành phần của mọi số từ `1` đến `N`; kiểm tra với `N = 99` bằng cách đếm tay: mỗi số `1..9` có 1 thành phần; trong `10..99` có 9 số tròn chục (1 thành phần) và 81 số còn lại (2 thành phần).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().decimalRepresentation(7009))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_components_todo(n):
    # TODO: đếm số chữ số khác 0 của n
    return 0

[7000, 9]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `7009` có chữ số `7, 0, 0, 9`. Bài 2: đọc các chữ số `3, 8, 0, 2, 5, 0` rồi nhân giá trị vị trí.

**Mức 2 (vừa):** Bài 3: dùng `sum(1 for ch in str(n) if ch != '0')` hoặc vòng `while n`. Bài 4: `[int(ch) * 10 ** (len(s) - 1 - i) for i, ch in enumerate(s) if ch != '0']` với `s = str(n)`. Bài 5: cộng `count_components(k)` cho `k = 1..N`.

**Mức 3 (gần đáp án):** Bài 1: `[7000, 9]`. Bài 2: `[300000, 80000, 200, 50]`. Bài 5: `total_components(99) = 9 + 9*1 + 81*2 = 180`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
assert Solution().decimalRepresentation(7009) == [7000, 9]

# Bài 2
assert Solution().decimalRepresentation(380250) == [300000, 80000, 200, 50]

# Bài 3
def count_components(n):
    return sum(1 for ch in str(n) if ch != '0')
assert count_components(537) == 3 and count_components(102) == 2 and count_components(10 ** 9) == 1
assert all(count_components(n) == len(Solution().decimalRepresentation(n)) for n in range(1, 3000))

# Bài 4
def decimal_representation_str(n):
    s = str(n)
    return [int(ch) * 10 ** (len(s) - 1 - i) for i, ch in enumerate(s) if ch != '0']
assert decimal_representation_str(537) == [500, 30, 7]
assert all(decimal_representation_str(n) == Solution().decimalRepresentation(n) for n in range(1, 3000))

# Bài 5
def total_components(N):
    return sum(count_components(k) for k in range(1, N + 1))
# 1..9: 9 số, mỗi số 1 thành phần = 9
# 10..99: 9 số tròn chục (1 thành phần) + 81 số còn lại (2 thành phần) = 9 + 162 = 171
assert total_components(9) == 9
assert total_components(99) == 9 + 9 + 81 * 2 == 180
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tách theo giá trị vị trí":**

```python
res, place = [], 1
while n:
    d = n % 10
    if d:                    # bỏ chữ số 0
        res.append(d * place)
    n //= 10
    place *= 10
res.reverse()                # đề cần giảm dần
return res
```

**Checklist trước khi submit:**

- [ ] Đã bỏ chữ số `0` (không có phần tử `0` trong kết quả)?
- [ ] Mỗi phần tử đã nhân đúng giá trị vị trí `10^pos`?
- [ ] Kết quả theo thứ tự **giảm dần** (đã đảo nếu bóc từ phải)?
- [ ] Thử `n = 1`, `n = 10`, `n = 102`, `n = 10^9` (biên của đề)?
- [ ] Chữ ký đúng `decimalRepresentation(self, n: int) -> List[int]` và đã `from typing import List`?
- [ ] `sum(res) == n` khi tự kiểm tra?